# Week 1 — Foundations: why accuracy lies

**Goal this week:** load the data, understand what we are predicting, and see for yourself why
"98% accuracy" can mean a model that catches **zero** criminals.

Run each cell with `Shift + Enter`. Read the text *before* running the code.
Cells marked **YOUR TURN** are for you to write — try before opening `solutions/week1_solutions.py`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from src.data import load_elliptic, time_split, feature_columns

DATA_DIR = ROOT / "data"
if not (DATA_DIR / "elliptic_txs_features.csv").exists():
    # No real data yet -> use the fake practice data (run: python tests/make_fake_data.py data_fake)
    DATA_DIR = ROOT / "data_fake"
    print("WARNING: using FAKE data. Numbers are meaningless. Put the real CSVs in data/.")
print("Reading from:", DATA_DIR)

## 2. Load the data

Two tables come back:
- `df` — one row per **transaction**: its id, its time step (1–49), its label, and 165 features.
- `edges` — one row per **money flow** from one transaction to another. This is the graph. We use it from Week 4.

Labels: `1` = illicit (fraud), `0` = licit (normal), `-1` = unknown (nobody labelled it).

In [ ]:
df, edges = load_elliptic(DATA_DIR)
print("transactions:", len(df))
print("money flows (edges):", len(edges))
print("features per transaction:", len(feature_columns(df)))
df.head()

**Check:** with the real data you should see about **203,769** transactions and **234,355** edges.
If not, a file is missing or in the wrong folder.

## 3. How rare is fraud?

This one number shapes every decision in the project.

In [ ]:
counts = df["label"].map({1: "illicit", 0: "licit", -1: "unknown"}).value_counts()
print(counts)
print()
print((counts / len(df) * 100).round(1).astype(str) + " %")

Most transactions are **unknown**. We cannot train or test on them, because we don't know the answer —
but they still sit in the graph and will matter for GNNs later.

Among the labelled ones, about **1 in 10** is illicit. Among *all* transactions, only about **2%**.

## 4. Split by time (never randomly)

We train on time steps **1–34** and test on **35–49**. That copies real life: you build a model on the past
and use it on the future. A random split would let the model "see the future" during training — that's
**data leakage**, and it makes scores look better than they really are.

In [ ]:
train, test = time_split(df)          # labelled transactions only
print("train:", len(train), "rows, illicit share =", round(train["label"].mean(), 3))
print("test: ", len(test),  "rows, illicit share =", round(test["label"].mean(), 3))
y_test = test["label"].values

## 5. The lazy model: predict "licit" for everything

This model does nothing. Let's see how good it *looks*.

Quick vocabulary (illicit = the "positive" class we want to catch):
- **Accuracy** — share of all predictions that are right.
- **Precision** — of the transactions I *flagged*, how many were really fraud?
- **Recall** — of all the *real* fraud, how much did I catch?
- **F1** — one number balancing precision and recall.

In [ ]:
lazy_pred = [0] * len(y_test)   # 0 = licit, for every single transaction

print("accuracy :", round(accuracy_score(y_test, lazy_pred), 3))
print("precision:", round(precision_score(y_test, lazy_pred, zero_division=0), 3))
print("recall   :", round(recall_score(y_test, lazy_pred), 3))
print("F1       :", round(f1_score(y_test, lazy_pred), 3))

High accuracy, **zero recall**. It caught no criminals at all.
This is why we will **never** report accuracy in this project. Our main metrics are illicit-class
**recall, precision and F1** (and PR-AUC from Week 3).

## 6. The confusion matrix

Every prediction falls in one of four boxes:

| | predicted licit | predicted illicit |
|---|---|---|
| **really licit** | true negative | false positive (false alarm) |
| **really illicit** | false negative (missed fraud) | true positive (caught) |

In [ ]:
cm = confusion_matrix(y_test, lazy_pred, labels=[0, 1])
pd.DataFrame(cm, index=["really licit", "really illicit"], columns=["pred licit", "pred illicit"])

## 7. YOUR TURN — a random guesser

Build a model that flags each test transaction as illicit **at random** with probability equal to the
illicit share in `train`. Then print accuracy, precision, recall and F1, and its confusion matrix.

Hints: `numpy.random.default_rng(42).random(len(y_test)) < p` gives you True/False with probability `p`.
Convert it to 0/1 with `.astype(int)`.

Before running, **predict** what recall you expect. Were you right?

In [ ]:
# YOUR CODE HERE


## 8. YOUR TURN — the opposite extreme

What happens if you flag **everything** as illicit? Compute the four metrics.
Which metric becomes perfect, and why is this model still useless?

In [ ]:
# YOUR CODE HERE


## 9. Self-check (answer aloud, then write in `notes/week1.md`)

1. Why is accuracy misleading when fraud is rare?
2. For a bank, which costs more: a missed fraud (false negative) or a false alarm (false positive)? When could the answer flip?
3. Why must we split by time instead of randomly?
4. Why can't we just treat the "unknown" transactions as licit?

**Next week:** EDA — how fraud changes across the 49 time steps, and the first look at the graph.